### IBL Brain-Wide Map: notes & data exploration
https://doi.org/10.1038/s41586-025-09235-0

In [1]:
# connect to the public OpenAlyx server
from one.api import ONE
from brainbox.io.one import SpikeSortingLoader
one = ONE(base_url='https://openalyx.internationalbrainlab.org')

In [2]:
one.load_cache(tag='Brainwidemap') 
sessions = one.search()  # get all sessions with the 'Brainwidemap' tag

Downloading: /Users/marilynjia/Downloads/ONE/openalyx.internationalbrainlab.org/Brainwidemap/tmppf5jmlof/cache.zip Bytes: 4353173


100%|██████████| 4.1515092849731445/4.1515092849731445 [00:01<00:00,  3.67it/s]


In [3]:
# this is the query that yields the few recordings for the Rhomboid Nucleus (RH) region
insertions_rh = one.search_insertions(atlas_acronym='RH', datasets='spikes.times.npy', project='brainwide')

# if we want to extend the search to all thalamic regions, we can do the following
insertions_th = one.search_insertions(atlas_acronym='TH', datasets='spikes.times.npy', project='brainwide')

# the Allen brain regions parcellation is hierarchical, and searching for Thalamus will return all child Rhomboid Nucleus (RH) regions
assert set(insertions_rh).issubset(set(insertions_th))

In [4]:
# find sessions that have spikes.times datasets
sessions_with_spikes = one.search(project='brainwide', datasets='spikes.times.npy')
print(sessions_with_spikes[0])

ebce500b-c530-47de-8cb1-963c552703ea


In [5]:
pid = 'da8dfec1-d265-44e8-84ce-6ae9c109b8bd'
sl = SpikeSortingLoader(pid=pid, one=one)
spikes, clusters, channels = sl.load_spike_sorting()
clusters = sl.merge_clusters(spikes, clusters, channels)
good = clusters['cluster_id'][clusters['label'] == 1]

eid, pname = one.pid2eid(pid)
trials = one.load_object(eid, 'trials')
wheel = one.load_object(eid, 'wheel')

(S3) /Users/marilynjia/Downloads/ONE/openalyx.internationalbrainlab.org/hoferlab/Subjects/SWC_043/2020-09-21/001/alf/probe00/pykilosort/#2024-05-06#/spikes.amps.npy: 100%|██████████| 319M/319M [00:10<00:00, 29.4MB/s] 
(S3) /Users/marilynjia/Downloads/ONE/openalyx.internationalbrainlab.org/hoferlab/Subjects/SWC_043/2020-09-21/001/alf/probe00/pykilosort/#2024-05-06#/spikes.clusters.npy: 100%|██████████| 159M/159M [00:05<00:00, 26.8MB/s] 
(S3) /Users/marilynjia/Downloads/ONE/openalyx.internationalbrainlab.org/hoferlab/Subjects/SWC_043/2020-09-21/001/alf/probe00/pykilosort/#2024-05-06#/spikes.depths.npy: 100%|██████████| 319M/319M [00:11<00:00, 26.6MB/s] 
(S3) /Users/marilynjia/Downloads/ONE/openalyx.internationalbrainlab.org/hoferlab/Subjects/SWC_043/2020-09-21/001/alf/probe00/pykilosort/#2024-05-06#/spikes.times.npy: 100%|██████████| 319M/319M [00:10<00:00, 30.1MB/s] 
/Users/marilynjia/miniforge3/envs/ntab_fa26/lib/python3.13/site-packages/one/util.py:464: ALFWarning: Multiple revisions:

In [ ]:
sl.raster(spikes, channels)